# RAG Evaluation Framework – Prototype 1

This notebook implements a minimal, controlled evaluation framework for
retrieval-augmented generation (RAG) systems.

We begin with a small synthetic corpus to:
- Create a controlled retrieval task
- Establish ground-truth mappings
- Enable systematic evaluation of retrieval accuracy and answer faithfulness

## Environment Setup

This cell loads environment variables and verifies that required configuration

In [4]:
from dotenv import load_dotenv
load_dotenv()

print("Environment variables loaded.")

Environment variables loaded.


## Step 1 — Define Controlled Corpus and Queries

We define:
- A small document corpus
- A set of queries
- Ground-truth document IDs for retrieval evaluation

In [5]:
import json
from pathlib import Path

# Base data path (relative to notebook)
BASE_DATA_PATH = Path("../data")
RAW_DATA_PATH = BASE_DATA_PATH / "raw"
PROCESSED_DATA_PATH = BASE_DATA_PATH / "processed"

with open(RAW_DATA_PATH / "documents.json") as f:
    documents = json.load(f)

with open(RAW_DATA_PATH / "queries.json") as f:
    queries = json.load(f)

print(documents[0])
print(queries[0])

{'id': 1, 'text': 'Anthropic develops AI systems with a focus on safety and alignment research.'}
{'query': 'What causes hallucinations in language models?', 'ground_truth_doc_id': 3}


## Step 2 — Document Embeddings

### Step 2.1 — Generate Document Embeddings

In this step, we generate vector embeddings for each document in our synthetic corpus using
the `EmbeddingModel` from `src/rag_eval/embeddings.py`. These embeddings will allow us
to perform similarity search and Retrieval-Augmented Generation (RAG) queries later.

The embeddings will first be attached to the `documents` in memory for inspection.

In [6]:
# Import our embedding abstraction
from rag_eval.embeddings import EmbeddingModel

# Initialize embedding model
embed_model = EmbeddingModel(model_name="text-embedding-3-small")

# Extract document texts
texts = [doc["text"] for doc in documents]

# Generate embeddings
embeddings_list = embed_model.embed_texts(texts)

# Attach embeddings to the documents (for inline inspection)
for doc, emb in zip(documents, embeddings_list):
    doc["embedding"] = emb

# Verify
documents[0]  # shows first doc with its embedding

{'id': 1,
 'text': 'Anthropic develops AI systems with a focus on safety and alignment research.',
 'embedding': [-0.006304657552391291,
  0.022708063945174217,
  0.05663835629820824,
  0.023800160735845566,
  -0.0037815391551703215,
  -0.017825014889240265,
  -0.010695008561015129,
  0.0483785979449749,
  0.033817317336797714,
  -0.01812628284096718,
  -0.010199172422289848,
  -0.0618603341281414,
  -0.012785054743289948,
  -0.04031968116760254,
  -0.01120967511087656,
  0.00340809253975749,
  -0.011046487838029861,
  -0.01743587665259838,
  0.026511570438742638,
  -0.03710615634918213,
  -0.017774803563952446,
  0.008837190456688404,
  -0.0006307795993052423,
  0.047951798886060715,
  0.016820788383483887,
  -0.06818696111440659,
  0.00822210218757391,
  0.014937864616513252,
  0.020900458097457886,
  0.0013745345640927553,
  0.05518222972750664,
  -0.02776685357093811,
  -0.01589187979698181,
  0.04235323891043663,
  -0.03228587284684181,
  0.037081051617860794,
  -0.021201726049184

### Step 2.2 — Persist Embeddings to Disk

To avoid recomputing embeddings each time the notebook is run, we save the embeddings
to disk in Parquet format under `data/processed/`. This allows us to quickly load
embeddings in future runs or other experiments.

In [7]:
from pathlib import Path
import pandas as pd

# Ensure processed data directory exists
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
embeddings_file = PROCESSED_DATA_PATH / "embeddings.parquet"

# Create DataFrame and save
df_embeddings = pd.DataFrame({
    "id": [doc["id"] for doc in documents],
    "text": texts,
    "embedding": embeddings_list
})

df_embeddings.to_parquet(embeddings_file, index=False)

print(f"Saved embeddings for {len(documents)} documents to {embeddings_file}")

Saved embeddings for 4 documents to ../data/processed/embeddings.parquet


### Step 2.3 — Validate Generated Embeddings

We validate that the embeddings were correctly generated and persisted without
assuming anything about the model used. The checks include:

1. The parquet file can be loaded successfully.
2. All expected columns exist: `id`, `text`, `embedding`.
3. Each embedding is a non-empty list.

This ensures that the persisted embeddings are usable for downstream retrieval and evaluation.

In [8]:
import pandas as pd
import numpy as np

# Load the persisted embeddings
embeddings_df = pd.read_parquet(embeddings_file)

# Check embeddings are non-empty sequences
for i, emb in enumerate(embeddings_df["embedding"]):
    if not isinstance(emb, (list, np.ndarray)):
        raise ValueError(f"Embedding at index {i} is not a list or ndarray")
    if len(emb) == 0:
        raise ValueError(f"Embedding at index {i} is empty")

print("✅ All embeddings are present and valid (non-empty sequences).")

✅ All embeddings are present and valid (non-empty sequences).


## Step 3 — Implement RAG Retrieval and Querying

In this phase, we will build a retrieval mechanism to support
Retrieval-Augmented Generation (RAG). This involves:

1. Creating a FAISS vector index from the document embeddings.
2. Running similarity search queries against the index.
3. Retrieving the most relevant documents for given user queries.

This step assumes that document embeddings have already been generated
and persisted in `data/processed/documents.parquet`.

### Step 3.1 — Load Document Embeddings and Build FAISS Index

In this step, we will:

1. Load the persisted document embeddings from `data_processed_path`.
2. Initialize a FAISS vector index.
3. Populate the index with the embeddings so that we can perform
   similarity searches in later steps.

In [9]:
import faiss
import numpy as np

# Use existing embeddings_df from earlier step
embedding_matrix = np.vstack(embeddings_df["embedding"].values).astype("float32")

# Initialize FAISS index (L2 distance)
embedding_dim = embedding_matrix.shape[1]
index = faiss.IndexFlatL2(embedding_dim)

# Add embeddings to index
index.add(embedding_matrix)

print(f"FAISS index created with {index.ntotal} vectors (dimension={embedding_dim}).")

FAISS index created with 4 vectors (dimension=1536).


### Step 3.2 — Query Embedding and Retrieval

Now that we have a FAISS index of all document embeddings, we can embed a new query and perform similarity search
to retrieve the most relevant document(s). Here we demonstrate embedding a single query using `EmbeddingModel.embed_text`
and then retrieving the top matching document from the FAISS index.

In [10]:
# Example query
query = "What causes hallucinations in language models?"

# Embed the query using the new embed_text method
query_embedding = embed_model.embed_text(query)

# Retrieve top-1 similar document
D, I = index.search(np.array([query_embedding], dtype=np.float32), k=1)

# Map index to document
retrieved_doc = embeddings_df.iloc[I[0][0]]

print(f"Query: {query}")
print(f"Retrieved document ID: {retrieved_doc['id']}")
print(f"Retrieved document text: {retrieved_doc['text']}")


Query: What causes hallucinations in language models?
Retrieved document ID: 3
Retrieved document text: Hallucinations in large language models occur when the model generates unsupported or fabricated information.
